# Muestra en Colab con Gemma 4 E4B — primera prueba sin OpenRouter

Copia independiente de `muestra_en_colab.ipynb`: mismo corpus, encoder bge-m3,
reranker y esquema oficial; decoder **google/gemma-4-E4B-it** convertido a GGUF Q4_0
por **ggml-org**. Solo texto: no se descargan los proyectores de imagen/audio.
Temperatura 0, semilla 42, razonamiento apagado y contexto de 8192 tokens.

**Preparación:**
1. Desde la raíz del proyecto actualizado:
   `.\.venv311\Scripts\python.exe -m src.index.paquete_colab --modo muestra --con-indice`.
   El paquete debe incluir los cambios de `src/config.py` y `src/generation/llm.py`.
2. Subir `dist/paquete_colab_muestra.zip` a **Mi unidad** de Google Drive.
3. Abrir este notebook en una **sesión nueva** de Colab, elegir **GPU T4** y ejecutar todo.
4. La celda 2 comienza con `RANGO = "1 3"`. Para las 50, dejar `RANGO = ""` e `IDS = ""`.

**OpenRouter está desactivado en todas las corridas, incluso con las 50 preguntas.**
No se cargan sus secretos, no se instalan las dependencias del juez y el ejecutor
rechaza el argumento `--ragas`. Su activación futura requiere un cambio explícito.
Gemma se ejecuta en proceso en Colab: no hay API de generación.

Referencias: [modelo original](https://huggingface.co/google/gemma-4-E4B-it),
[conversión y archivo Q4_0](https://huggingface.co/ggml-org/gemma-4-E4B-it-GGUF/blob/main/gemma-4-E4B-it-Q4_0.gguf).
Los ~4.59 GB corresponden al archivo; la GPU también aloja la caché y los otros modelos.
Esta corrida es experimental: E4B significa parámetros efectivos, no tamaño total.

In [ ]:
# 1. GPU y ejecutor: los fallos detienen la celda; no se ocultan detrás de un tick verde.
import os
import subprocess
import sys
os.environ.update({"PYTHONIOENCODING": "utf-8", "JAX_PLATFORMS": "cpu",
                   "XLA_PYTHON_CLIENT_PREALLOCATE": "false", "USE_TF": "0", "USE_JAX": "0"})
os.environ.pop("OPENROUTER_API_KEY", None)

def ejecutar(argumentos, *, env_extra=None):
    argumentos = [str(a) for a in argumentos]
    if any(a == "--ragas" or a.startswith("--ragas=") for a in argumentos):
        raise RuntimeError("RAGAS/OpenRouter no está autorizado en este notebook.")
    entorno = os.environ.copy()
    entorno.update(env_extra or {})
    entorno.pop("OPENROUTER_API_KEY", None)
    entorno["PYTHONUNBUFFERED"] = "1"
    print("Iniciando proceso; los mensajes aparecerán aquí…", flush=True)
    proceso = subprocess.Popen(
        argumentos, env=entorno, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace",
        bufsize=1, start_new_session=(os.name == "posix"))
    try:
        for linea in proceso.stdout:
            print(linea, end="", flush=True)
        codigo = proceso.wait()
    except BaseException:
        if proceso.poll() is None:
            if os.name == "posix":
                import signal
                try:
                    os.killpg(proceso.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
            else:
                proceso.terminate()
            try:
                proceso.wait(timeout=5)
            except subprocess.TimeoutExpired:
                if os.name == "posix":
                    os.killpg(proceso.pid, signal.SIGKILL)
                else:
                    proceso.kill()
                proceso.wait(timeout=5)
        raise
    finally:
        proceso.stdout.close()
    if codigo:
        raise subprocess.CalledProcessError(codigo, argumentos)


ejecutar(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])
ejecutar([sys.executable, "-c", "import torch; assert torch.cuda.is_available(), 'Seleccione una GPU en Colab'; print('GPU:', torch.cuda.get_device_name(0))"])

## 2. Paquete, selección de preguntas y carpeta independiente

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import re
import shutil
import zipfile
from google.colab import drive
drive.mount("/content/drive")

RUTA_ZIP = "/content/drive/MyDrive/paquete_colab_muestra.zip"
TAG = "gemma_e4b_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RANGO = "1 3"  # Primera prueba. "" = las 50; "1 15" = selección múltiple.
IDS = ""       # Alternativa: ids separados por comas; no combinar con RANGO.

if not re.fullmatch(r"[A-Za-z0-9_-]+", TAG):
    raise ValueError("TAG solo admite letras, números, guion y guion bajo.")
if RANGO and IDS:
    raise ValueError("Use RANGO o IDS, no ambos.")
ARG_PREGUNTAS = []
if RANGO:
    posiciones = [int(x) for x in RANGO.split()]
    if len(posiciones) != 2 or not 1 <= posiciones[0] <= posiciones[1] <= 50:
        raise ValueError("RANGO debe ser INICIO FIN dentro de las 50 preguntas.")
    ARG_PREGUNTAS = ["--rango", *map(str, posiciones)]
if IDS:
    if not all(x.strip().isdigit() for x in IDS.split(",")):
        raise ValueError("IDS debe contener ids numéricos separados por comas.")
    ARG_PREGUNTAS = ["--ids", IDS]
PARCIAL = bool(ARG_PREGUNTAS)

PROYECTO = Path("/content/proyecto_gemma")
PROYECTO.mkdir(parents=True, exist_ok=True)
if (PROYECTO / ".env").exists() or (PROYECTO / "scripts/.env").exists():
    raise RuntimeError("Esta carpeta contiene un .env; use una sesión nueva sin secretos del juez.")
with zipfile.ZipFile(RUTA_ZIP) as paquete:
    for nombre in paquete.namelist():
        destino = (PROYECTO / nombre).resolve()
        if not destino.is_relative_to(PROYECTO.resolve()) or ".env" in Path(nombre).parts:
            raise ValueError(f"Archivo no permitido en el paquete: {nombre}")
    for necesario in ("src/config.py", "src/generation/llm.py", "data/sample_50.jsonl",
                      "schema/submission.schema.json", "build/indice/chunks.jsonl"):
        if necesario not in paquete.namelist():
            raise ValueError(f"Paquete incompleto: falta {necesario}")
    paquete.extractall(PROYECTO)
os.chdir(PROYECTO)
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))
DIR_RUN = PROYECTO / "runs" / TAG
RESPALDO = Path("/content/drive/MyDrive/experimentos_gemma") / TAG
print("Proyecto:", PROYECTO, "\nCorrida:", TAG, "\nPreguntas:", ARG_PREGUNTAS or "las 50")

## 3. Dependencias con CUDA

Se conserva el PyTorch de Colab. Se instala llama-cpp-python 0.3.35 con CUDA
desde su índice exclusivo (no se permite seleccionar la rueda CPU desde PyPI).
La celda 3b compila con CUDA **solo si falta una rueda funcional**. Muestra los
mensajes de compilación y el tiempo transcurrido, compila únicamente para la GPU
asignada y se detiene después de 30 minutos. Al interrumpirla, también detiene sus
procesos de compilación. No instala RAGAS.


In [ ]:
ejecutar([sys.executable, "-m", "pip", "install", "-q",
          "sentence-transformers==6.1.0", "transformers==5.17.0", "faiss-cpu==1.15.1",
          "bm25s==0.3.11", "PyStemmer==3.1.0", "python-dotenv==1.2.3",
          "jsonschema==4.26.0", "httpx", "huggingface_hub==1.33.0",
          "diskcache>=5.6.1", "jinja2"])
COMPILAR_LLAMA = False
try:
    ejecutar([sys.executable, "-m", "pip", "install", "-v", "--force-reinstall", "--no-deps",
              "--only-binary=llama-cpp-python", "llama-cpp-python==0.3.35",
              "--index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu124"])
    ejecutar([sys.executable, "-c", "import llama_cpp; assert llama_cpp.llama_supports_gpu_offload(), 'La rueda no tiene CUDA'"])
except subprocess.CalledProcessError:
    COMPILAR_LLAMA = True
    print("Se compilará llama.cpp con CUDA en la celda 3b.")

In [ ]:
# 3b. Instalar la rueda CUDA oficial: esta celda no compila código.
import glob
import os
import platform
import site
import subprocess
import sys

if platform.system() != "Linux" or platform.machine() != "x86_64":
    raise RuntimeError("Esta celda está preparada para el entorno Linux de Google Colab.")

entorno = os.environ.copy()
entorno.pop("OPENROUTER_API_KEY", None)
subprocess.run(
    [sys.executable, "-c", "import torch; assert torch.cuda.is_available(), 'Activa GPU en Colab'; "
     "print('GPU:', torch.cuda.get_device_name(0), '| CUDA de PyTorch:', torch.version.cuda)"],
    env=entorno, check=True, timeout=60)

rueda = (
    "https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.35-cu124/"
    "llama_cpp_python-0.3.35-py3-none-manylinux_2_35_x86_64.whl"
    "#sha256=62fc788a4ecee5a579f40708ccfe0c4d566dd5c9959497bdc0544dbba94d82aa"
)
print("Descargando rueda CUDA ya compilada (~1,68 GB). No se ejecutará CMake ni nvcc.", flush=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps",
     "--only-binary=:all:", "--timeout", "60", "--retries", "3", rueda],
    env=entorno, check=True, timeout=900)

# Hacer visibles las bibliotecas CUDA a todos los procesos posteriores del notebook.
rutas = []
for base in site.getsitepackages():
    for patron in ("nvidia/*/lib", "torch/lib", "llama_cpp/lib", "llama_cpp_python.libs"):
        rutas.extend(glob.glob(os.path.join(base, patron)))
rutas.extend(glob.glob("/usr/local/cuda*/lib64"))
rutas.extend(p for p in os.environ.get("LD_LIBRARY_PATH", "").split(":") if p)
os.environ["LD_LIBRARY_PATH"] = ":".join(dict.fromkeys(rutas))
entorno["LD_LIBRARY_PATH"] = os.environ["LD_LIBRARY_PATH"]

# Comprobar en un proceso nuevo, sin descargar ni cargar los pesos de Gemma.
subprocess.run(
    [sys.executable, "-c", "import torch, llama_cpp; "
     "assert torch.cuda.is_available(), 'PyTorch no detecta GPU'; "
     "assert llama_cpp.llama_supports_gpu_offload(), 'La rueda no pudo activar CUDA; copia este error'; "
     "print('llama-cpp-python:', llama_cpp.__version__); print('CUDA OK: puedes ejecutar 3c')"],
    env=entorno, check=True, timeout=90)
COMPILAR_LLAMA = False


In [ ]:
# 3c. No se permite pasar silenciosamente a CPU.
ejecutar([sys.executable, "-c", "import torch, llama_cpp; print('PyTorch CUDA:', torch.cuda.is_available()); print('llama.cpp CUDA:', llama_cpp.llama_supports_gpu_offload()); assert torch.cuda.is_available() and llama_cpp.llama_supports_gpu_offload()"])

## 4. Configurar Gemma y verificar/reutilizar el índice

Solo cambia el decoder. Un índice FAISS válido se reutiliza sin recalcular embeddings
ni reconstruir BM25. Si falta o cambió, se actualiza con el encoder en GPU.
La configuración y los resultados se respaldan en Drive para poder reanudar con el mismo TAG.

In [ ]:
import json
from dataclasses import fields
DECODER_GGUF_REPO = "ggml-org/gemma-4-E4B-it-GGUF"
DECODER_GGUF_FILE = "gemma-4-E4B-it-Q4_0.gguf"
os.environ.update({"DECODER_GGUF_REPO": DECODER_GGUF_REPO,
                   "DECODER_GGUF_FILE": DECODER_GGUF_FILE,
                   "DECODER_MODEL": "google/gemma-4-E4B-it",
                   "ENCODER_DEVICE": "cuda", "ENCODER_BATCH_SIZE": "16",
                   "DECODER_DEVICE": "cuda", "DECODER_BACKEND": "llamacpp",
                   "DECODER_NUM_CTX": "8192", "DECODER_THREADS": "4",
                   "DECODER_TEMPERATURE": "0", "DECODER_SEED": "42",
                   "MC_RAZONAMIENTO_TOKENS": "0", "CORPUS_SOURCE": "local",
                   "CORPUS_OUT_DIR": "./build/corpus", "INDEX_DIR": "./build/indice",
                   "OMP_NUM_THREADS": "4", "OPENROUTER_API_KEY": ""})
from src.config import cargar
cfg = cargar()
try:
    cfg.validar_final()
except ValueError as exc:
    raise RuntimeError("Paquete antiguo: regenere el zip desde el proyecto con soporte de Gemma.") from exc
from src.generation.llm import LLM
if "_gemma_formateador" not in LLM.contar_tokens.__code__.co_consts and "_gemma_formateador" not in LLM.contar_tokens.__code__.co_names:
    raise RuntimeError("El paquete no incluye la adaptación de la plantilla Gemma; regenere el zip.")

parametros = {f.name: getattr(cfg, f.name) for f in fields(cfg)
              if f.name not in {"openrouter_api_key", "hf_token", "team_name", "corpus_zip_url", "corpus_zip_sha256"}
              and not isinstance(getattr(cfg, f.name), Path)}
identidad = {"modelo_original": "google/gemma-4-E4B-it", "parametros": parametros,
             "openrouter": False, "ragas": False}
for carpeta in (DIR_RUN, RESPALDO):
    previo = carpeta / "experimento.json"
    if previo.exists() and json.loads(previo.read_text(encoding="utf-8")) != identidad:
        raise RuntimeError("Ese TAG corresponde a otra configuración; elija uno nuevo.")
    if (carpeta / "submissions.jsonl").exists() and not previo.exists():
        raise RuntimeError("El TAG tiene respuestas sin identidad de modelo; elija uno nuevo.")
if RESPALDO.exists():
    shutil.copytree(RESPALDO, DIR_RUN, dirs_exist_ok=True)
DIR_RUN.mkdir(parents=True, exist_ok=True)
(DIR_RUN / "experimento.json").write_text(json.dumps(identidad, ensure_ascii=False, indent=2), encoding="utf-8")
shutil.copytree(DIR_RUN, RESPALDO, dirs_exist_ok=True)

import hashlib
def sha256_archivo(ruta):
    h = hashlib.sha256()
    with ruta.open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()
d = cfg.index_dir
manifiesto = d / "index_manifest.json"
necesarios = [d / "index.faiss", manifiesto, d / "chunks.jsonl"] + [
    d / "bm25" / nombre for nombre in ("params.index.json", "vocab.index.json",
    "data.csc.index.npy", "indices.csc.index.npy", "indptr.csc.index.npy")]
reutilizar = False
if all(p.is_file() for p in necesarios):
    m = json.loads(manifiesto.read_text(encoding="utf-8"))
    reutilizar = (m.get("modelo") == cfg.encoder_model
                  and m.get("sha256_chunks") == sha256_archivo(d / "chunks.jsonl")
                  and m.get("sha256_faiss") == sha256_archivo(d / "index.faiss"))
if reutilizar:
    print("Índice reutilizado (sin cambios):", m["n_fragmentos"], "fragmentos.")
else:
    ejecutar([sys.executable, "-m", "src.index.build"])
    if not all(p.is_file() for p in necesarios):
        raise RuntimeError("La construcción no dejó un índice completo.")
print(manifiesto.read_text(encoding="utf-8"))

In [ ]:
# 4b. Modelo real: prueba JSON sin caché antes de responder la muestra.
# La carga verifica arquitectura gemma4, plantilla nativa y fin de turno.
ejecutar([sys.executable, "-m", "src.generation.ping", "--sin-cache"])

## 5. Responder las preguntas seleccionadas

El proceso usa Gemma con JSON forzado. Se comprueba el esquema oficial, que los ids
coincidan con la selección y que no haya errores de generación/esquema.
Las respuestas terminadas se copian a Drive al terminar o interrumpir el proceso.
Para reanudar en otra sesión, vuelva a usar el **mismo TAG y configuración** de la celda 2.

In [ ]:
try:
    ejecutar([sys.executable, "-m", "src.pipeline.main", "--split", "sample",
              "--tag", TAG, *ARG_PREGUNTAS])
finally:
    # Guarda los resultados parciales incluso si el proceso falla o se interrumpe.
    if DIR_RUN.exists():
        shutil.copytree(DIR_RUN, RESPALDO, dirs_exist_ok=True)

resumen = json.loads((DIR_RUN / "tiempos.json").read_text(encoding="utf-8"))
print(json.dumps(resumen, ensure_ascii=False, indent=2))
if resumen.get("errores_esquema", 0):
    raise RuntimeError("La corrida tuvo errores: revise trazas.jsonl antes de evaluar.")
import jsonschema
from src.pipeline.main import leer_jsonl, rango
preguntas = leer_jsonl(Path("data/sample_50.jsonl"))
seleccion = rango(preguntas, *posiciones) if RANGO else preguntas
if IDS:
    ids = {int(x) for x in IDS.split(",")}
    if ids - {q["id"] for q in preguntas}:
        raise ValueError("Hay ids que no existen en la muestra.")
    seleccion = [q for q in preguntas if q["id"] in ids]
respuestas = leer_jsonl(DIR_RUN / "submissions.jsonl")
esquema = json.loads(Path("schema/submission.schema.json").read_text(encoding="utf-8"))
for respuesta in respuestas:
    jsonschema.validate(respuesta, esquema)
ids_respuesta = [r["id"] for r in respuestas]
if len(ids_respuesta) != len(set(ids_respuesta)):
    raise RuntimeError("La entrega tiene ids duplicados.")
if set(ids_respuesta) != {q["id"] for q in seleccion}:
    raise RuntimeError("La entrega no coincide con la selección; use un TAG nuevo si redujo el rango.")
print("Validación correcta:", len(respuestas), "respuestas. Sin OpenRouter.")

## 6. Evaluación sin juez: sin costo de OpenRouter

In [ ]:
if PARCIAL:
    oro = {q["id"]: q for q in preguntas}
    mc = [r for r in respuestas if r["formato"] == "multiple_choice"]
    aciertos = sum(r.get("respuesta_correcta") == oro[r["id"]]["respuesta_correcta"] for r in mc)
    reporte = {"parcial": True, "n_respuestas": len(respuestas), "mc_aciertos": aciertos,
               "mc_total": len(mc), "ragas": "no ejecutado", "openrouter": False}
    (DIR_RUN / "reporte_parcial.json").write_text(json.dumps(reporte, indent=2), encoding="utf-8")
    print(f"Selección múltiple: {aciertos}/{len(mc)}. Sin evaluación oficial completa por ser parcial.")
else:
    ejecutar([sys.executable, "scripts/evaluate.py", "--submission", DIR_RUN / "submissions.jsonl",
              "--split", "sample", "--out", DIR_RUN / "reporte.json"])
shutil.copytree(DIR_RUN, RESPALDO, dirs_exist_ok=True)

## 7. Secretos de OpenRouter: desactivados

Esta prueba no solicita ni lee la llave del juez.

In [ ]:
os.environ.pop("OPENROUTER_API_KEY", None)
print("No se cargó ninguna llave de OpenRouter.")

## 8. RAGAS: desactivado

Se habilitará únicamente tras una nueva indicación del usuario.

In [ ]:
print("RAGAS omitido. Las 50 preguntas también se evalúan sin juez en esta versión.")

## 9. Guardar índice y resultados

El zip incluye `runs/<TAG>/`, los fragmentos e índice completos y la caché de embeddings.
Se copia a Drive y se ofrece su descarga. No incluye modelos, secretos ni `.env`.
Los resultados de Gemma quedan identificados por su propio TAG y `experimento.json`.

In [ ]:
from google.colab import files
shutil.copytree(DIR_RUN, RESPALDO, dirs_exist_ok=True)
archivo_zip = Path("/content") / f"{TAG}.zip"
carpetas = [DIR_RUN, PROYECTO / "build/indice", PROYECTO / "build/cache/emb"]
with zipfile.ZipFile(archivo_zip, "w", zipfile.ZIP_DEFLATED) as salida:
    for carpeta in carpetas:
        if carpeta.exists():
            for archivo in sorted(carpeta.rglob("*")):
                if archivo.is_file() and archivo.name != ".env":
                    salida.write(archivo, archivo.relative_to(PROYECTO).as_posix())
destino_drive = Path("/content/drive/MyDrive/experimentos_gemma") / archivo_zip.name
shutil.copy2(archivo_zip, destino_drive)
print("Zip guardado en Drive:", destino_drive)
files.download(str(archivo_zip))